# Purpose

This notebook redraws the DQN learning curve for the thesis report in a publication-quality style.

The full learning curve is preserved, but a zoomed version is also generated because a few catastrophic exploration episodes can produce extremely negative rewards and distort the y-axis. The notebook only reads TensorBoard logs and saves figures; it does not retrain the model or modify simulation, training, evaluation, or configuration code.

# Load TensorBoard scalar data

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LOGS_DIR = PROJECT_ROOT / "logs"
PREFERRED_LOG_DIR = LOGS_DIR / "dqn_ran_slicing_300k"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures_paper"
SCALAR_TAG = "rollout/ep_rew_mean"
SMOOTHING_WINDOW = 10

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "black",
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "font.size": 11,
    }
)

def find_event_files():
    """Find event files from the preferred 300k run, then fall back to latest 300k-like logs."""
    if PREFERRED_LOG_DIR.exists():
        event_files = sorted(PREFERRED_LOG_DIR.rglob("events.out.tfevents.*"))
        if event_files:
            return event_files, PREFERRED_LOG_DIR

    all_event_files = sorted(LOGS_DIR.rglob("events.out.tfevents.*"), key=lambda p: p.stat().st_mtime, reverse=True)
    candidate_300k = [p for p in all_event_files if "300k" in str(p).lower()]
    if candidate_300k:
        latest_parent = candidate_300k[0].parent
        return sorted(latest_parent.rglob("events.out.tfevents.*")), latest_parent

    if all_event_files:
        latest_parent = all_event_files[0].parent
        return sorted(latest_parent.rglob("events.out.tfevents.*")), latest_parent

    raise FileNotFoundError(f"No TensorBoard event files found under {LOGS_DIR}")

def load_scalar_events(event_files, tag):
    """Load the requested scalar tag from all selected TensorBoard event files."""
    rows = []
    available_tags = set()

    for event_file in event_files:
        accumulator = EventAccumulator(str(event_file))
        accumulator.Reload()
        scalar_tags = accumulator.Tags().get("scalars", [])
        available_tags.update(scalar_tags)

        if tag not in scalar_tags:
            continue

        for event in accumulator.Scalars(tag):
            rows.append({"step": event.step, "reward": event.value})

    if not rows:
        print(f"Scalar tag not found: {tag}")
        print("Available scalar tags:")
        for scalar_tag in sorted(available_tags):
            print(f"- {scalar_tag}")
        raise SystemExit("Required TensorBoard scalar tag is missing; stopping notebook execution.")

    df = pd.DataFrame(rows)
    df = df.sort_values("step").drop_duplicates(subset="step", keep="last").reset_index(drop=True)
    df["reward_smooth"] = df["reward"].rolling(window=SMOOTHING_WINDOW, min_periods=1, center=True).mean()
    return df

event_files, selected_log_dir = find_event_files()
print(f"Selected log directory: {selected_log_dir}")
print("Event files:")
for event_file in event_files:
    print(f"- {event_file}")

reward_df = load_scalar_events(event_files, SCALAR_TAG)
reward_df.head()

Selected log directory: D:\Projects\dqn-ran-slicing-oran-iov\logs\dqn_ran_slicing_300k
Event files:
- D:\Projects\dqn-ran-slicing-oran-iov\logs\dqn_ran_slicing_300k\dqn_ran_slicing_1\events.out.tfevents.1780634481.DUYANH.10920.0


,step,reward,reward_smooth
0,800,-220.510513,-195.570627
1,1600,-191.340683,-194.540166
2,2400,-187.908325,-193.885679
3,3200,-176.167313,-196.516382
4,4000,-201.926300,-199.024914


# Inspect reward statistics

In [2]:
stats = {
    "number of points": len(reward_df),
    "min reward": reward_df["reward"].min(),
    "max reward": reward_df["reward"].max(),
    "mean reward": reward_df["reward"].mean(),
    "median reward": reward_df["reward"].median(),
    "final reward value": reward_df["reward"].iloc[-1],
    "final smoothed reward value": reward_df["reward_smooth"].iloc[-1],
}

for name, value in stats.items():
    print(f"{name}: {value}")

number of points: 375
min reward: -190093.21875
max reward: 67.9149169921875
mean reward: -12657.735132315
median reward: 46.776329040527344
final reward value: 58.444786071777344
final smoothed reward value: 54.82114791870117


# Plot full-scale learning curve

In [3]:
def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.5)

def save_figure(fig, filename):
    path = FIGURES_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=300)
    plt.close(fig)
    return path

fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.plot(reward_df["step"], reward_df["reward"], linewidth=0.8, alpha=0.35, label="Raw episode reward")
ax.plot(reward_df["step"], reward_df["reward_smooth"], linewidth=2.0, label="Smoothed episode reward")
ax.axvline(300000, linestyle="--", linewidth=1.2, label="300000 timesteps")
ax.set_xlabel("Training timesteps")
ax.set_ylabel("Episode reward")
ax.set_title("Learning curve of DQN-based RAN slicing")
ax.legend()
clean_axes(ax)
full_curve_path = save_figure(fig, "fig_learning_curve_dqn_full.png")
full_curve_path

WindowsPath('D:/Projects/dqn-ran-slicing-oran-iov/results/figures_paper/fig_learning_curve_dqn_full.png')

# Plot zoomed learning curve for convergence analysis

In [4]:
ZOOM_Y_LIMITS = (-500, 150)

fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.plot(reward_df["step"], reward_df["reward"], linewidth=0.8, alpha=0.35, label="Raw episode reward")
ax.plot(reward_df["step"], reward_df["reward_smooth"], linewidth=2.0, label="Smoothed episode reward")
ax.axvline(300000, linestyle="--", linewidth=1.2, label="300000 timesteps")
ax.set_ylim(*ZOOM_Y_LIMITS)
ax.set_xlabel("Training timesteps")
ax.set_ylabel("Episode reward")
ax.set_title("Zoomed learning curve of DQN-based RAN slicing")
ax.legend()
clean_axes(ax)
zoom_curve_path = save_figure(fig, "fig_learning_curve_dqn_zoom.png")
zoom_curve_path

WindowsPath('D:/Projects/dqn-ran-slicing-oran-iov/results/figures_paper/fig_learning_curve_dqn_zoom.png')

The zoomed view is used because a few catastrophic exploration episodes produce very large negative rewards. The outlier data are not deleted or altered; the displayed y-axis range is limited only to make the convergence region easier to inspect.

# Save publication-quality figures

In [5]:
REPORT_Y_LIMITS = (-200, 120)

fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.plot(reward_df["step"], reward_df["reward_smooth"], linewidth=2.2, label="Smoothed episode reward")
ax.axvline(300000, linestyle="--", linewidth=1.2, label="300000 timesteps")
ax.set_ylim(*REPORT_Y_LIMITS)
ax.set_xlabel("Training timesteps")
ax.set_ylabel("Episode reward")
ax.set_title("Learning curve of DQN-based RAN slicing")
ax.legend()
clean_axes(ax)
report_curve_path = save_figure(fig, "fig_learning_curve_dqn_report.png")
report_curve_path

WindowsPath('D:/Projects/dqn-ran-slicing-oran-iov/results/figures_paper/fig_learning_curve_dqn_report.png')

# Clean report-ready learning curve

In [7]:
CLEAN_REPORT_Y_LIMITS = (-150, 100)
CLEAN_REPORT_X_LIMITS = (0, 300000)

# The original reward_df is not modified. This display-only series hides very low
# smoothed values so clipped outliers do not appear as misleading vertical drops.
reward_smooth_display = reward_df["reward_smooth"].where(reward_df["reward_smooth"] >= -150, np.nan)

fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.plot(reward_df["step"], reward_smooth_display, linewidth=2.2, label="Smoothed episode reward")
ax.axvline(300000, linestyle="--", linewidth=1.2, label="300000 timesteps")
ax.set_xlim(*CLEAN_REPORT_X_LIMITS)
ax.set_ylim(*CLEAN_REPORT_Y_LIMITS)
ax.set_xlabel("Training timesteps")
ax.set_ylabel("Smoothed episode reward")
ax.set_title("Learning curve of DQN-based RAN slicing")
ax.legend()
clean_axes(ax)
report_clean_curve_path = save_figure(fig, "fig_learning_curve_dqn_report_clean.png")
report_clean_curve_path

WindowsPath('D:/Projects/dqn-ran-slicing-oran-iov/results/figures_paper/fig_learning_curve_dqn_report_clean.png')

The curve is shown in a zoomed range to highlight the convergence trend. A few extremely negative rewards during early exploration are omitted from the displayed range for readability, but the original data are not modified.

# Interpretation

Early exploration can produce very low rewards because the agent may choose PRB allocations that severely violate the Ambulance Slice latency requirement. These outliers distort the full-scale plot, so the zoomed curve is used to inspect convergence behavior more clearly while preserving the original data. The smoothed reward should be interpreted cautiously; if the plotted curve supports it, it may indicate that the reward tends to stabilize after training, but this does not imply that the learned policy is globally optimal.

# Validation

In [8]:
saved_paths = [full_curve_path, zoom_curve_path, report_curve_path, report_clean_curve_path]
for path in saved_paths:
    print(f"{path}: exists={path.exists()}")

D:\Projects\dqn-ran-slicing-oran-iov\results\figures_paper\fig_learning_curve_dqn_full.png: exists=True
D:\Projects\dqn-ran-slicing-oran-iov\results\figures_paper\fig_learning_curve_dqn_zoom.png: exists=True
D:\Projects\dqn-ran-slicing-oran-iov\results\figures_paper\fig_learning_curve_dqn_report.png: exists=True
D:\Projects\dqn-ran-slicing-oran-iov\results\figures_paper\fig_learning_curve_dqn_report_clean.png: exists=True
